# 03b — Panic and Stabilization Threshold Sensitivity Analysis

This notebook evaluates whether the identification of panic episodes is sensitive to alternative threshold choices.

The finalized EGARCH conditional-volatility estimates and structural-break regimes are held fixed. Only the thresholds used to classify panic and stabilization are varied. Therefore, this analysis evaluate the robustness of the episode classification stage.

The baseline specification is:

**Panic threshold:** 50% increase in mean regime volatility

**Stabilization threshold:** 25% decline from peak regime volatility

The sensitivity analysis considers:

**Panic thresholds:** 40%, 50%, and 60%

**Stabilization thresholds:** 20%, 25%, and 30%

In [ ]:
from itertools import product

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

Imports complete.


## Load Finalized Multi-Asset Regimes

In [ ]:
# Load finalized volatility regimes and baseline episode dataset

multi_regimes = pd.read_csv(
    "data/processed/volatility_regimes_23_assets_2000_2026.csv",
    parse_dates=["StartDate", "EndDate"]
)

baseline_episodes = pd.read_csv(
    "data/processed/panic_stabilization_episodes_23_assets_2000_2026.csv",
    parse_dates=["PanicStart", "StabilizationDate"]
)

print("Regime rows:", len(multi_regimes))
print("Baseline episodes:", len(baseline_episodes))
print("Assets:", multi_regimes["Asset"].nunique())

Regime rows: 588
Baseline episodes: 108
Assets: 23


In [ ]:
print("Regime columns:")
print(multi_regimes.columns.tolist())

print("\nEpisode columns:")
print(baseline_episodes.columns.tolist())

Regime columns:
['Asset', 'Regime', 'StartIndex', 'EndIndex', 'StartDate', 'EndDate', 'MeanVolatility']

Episode columns:
['Asset', 'PanicStart', 'StabilizationDate', 'PanicStartVolatility', 'PeakVolatility', 'RecoveryDays']


In [ ]:
# Apply the same episode-identification logic used in the baseline analysis

def identify_panic_episodes(
    regimes,
    panic_threshold=0.50,
    stabilization_threshold=0.25
):
    episodes = []

    in_panic = False
    panic_start = None
    panic_start_index = None
    panic_initial_volatility = None
    peak_volatility = None

    regimes = (
        regimes
        .sort_values("Regime")
        .reset_index(drop=True)
        .copy()
    )

    for i in range(1, len(regimes)):

        previous_mean = regimes.loc[i - 1, "MeanVolatility"]
        current_mean = regimes.loc[i, "MeanVolatility"]

        if not in_panic:

            relative_increase = (
                (current_mean - previous_mean)
                / previous_mean
            )

            if relative_increase >= panic_threshold:

                in_panic = True

                panic_start = regimes.loc[i, "StartDate"]
                panic_start_index = regimes.loc[i, "StartIndex"]

                panic_initial_volatility = current_mean
                peak_volatility = current_mean

        else:

            if current_mean > peak_volatility:
                peak_volatility = current_mean

            decline_from_peak = (
                (peak_volatility - current_mean)
                / peak_volatility
            )

            if decline_from_peak >= stabilization_threshold:

                stabilization_date = regimes.loc[i, "StartDate"]
                stabilization_index = regimes.loc[i, "StartIndex"]

                episodes.append({
                    "Asset": regimes.loc[i, "Asset"],
                    "PanicStart": panic_start,
                    "StabilizationDate": stabilization_date,
                    "PanicStartVolatility": panic_initial_volatility,
                    "PeakVolatility": peak_volatility,
                    "RecoveryDays": int(
                        stabilization_index - panic_start_index
                    )
                })

                in_panic = False
                panic_start = None
                panic_start_index = None
                panic_initial_volatility = None
                peak_volatility = None

    return pd.DataFrame(episodes)

In [ ]:
# Define the sensitivity grid

panic_thresholds = [0.40, 0.50, 0.60]
stabilization_thresholds = [0.20, 0.25, 0.30]

threshold_combinations = list(
    product(
        panic_thresholds,
        stabilization_thresholds
    )
)

threshold_combinations

[(0.4, 0.2),
 (0.4, 0.25),
 (0.4, 0.3),
 (0.5, 0.2),
 (0.5, 0.25),
 (0.5, 0.3),
 (0.6, 0.2),
 (0.6, 0.25),
 (0.6, 0.3)]

In [ ]:
# Run all 9 threshold specifications across all 23 assets

sensitivity_summaries = []
sensitivity_episodes = []

for panic_threshold, stabilization_threshold in threshold_combinations:

    specification_episodes = []

    for asset in sorted(multi_regimes["Asset"].unique()):

        asset_regimes = (
            multi_regimes[
                multi_regimes["Asset"] == asset
            ]
            .copy()
            .sort_values("Regime")
            .reset_index(drop=True)
        )

        episodes = identify_panic_episodes(
            regimes=asset_regimes,
            panic_threshold=panic_threshold,
            stabilization_threshold=stabilization_threshold
        )

        if not episodes.empty:

            episodes["PanicThreshold"] = panic_threshold
            episodes["StabilizationThreshold"] = stabilization_threshold

            specification_episodes.append(episodes)

    if specification_episodes:

        specification_df = pd.concat(
            specification_episodes,
            ignore_index=True
        )

        sensitivity_episodes.append(specification_df)

        sensitivity_summaries.append({
            "PanicThreshold": panic_threshold,
            "StabilizationThreshold": stabilization_threshold,
            "NumberOfEpisodes": len(specification_df),
            "AssetsWithEpisodes": specification_df["Asset"].nunique(),
            "MeanRecoveryDays": specification_df["RecoveryDays"].mean(),
            "MedianRecoveryDays": specification_df["RecoveryDays"].median(),
            "MinRecoveryDays": specification_df["RecoveryDays"].min(),
            "MaxRecoveryDays": specification_df["RecoveryDays"].max()
        })

    else:

        sensitivity_summaries.append({
            "PanicThreshold": panic_threshold,
            "StabilizationThreshold": stabilization_threshold,
            "NumberOfEpisodes": 0,
            "AssetsWithEpisodes": 0,
            "MeanRecoveryDays": np.nan,
            "MedianRecoveryDays": np.nan,
            "MinRecoveryDays": np.nan,
            "MaxRecoveryDays": np.nan
        })


multi_sensitivity_summary = pd.DataFrame(
    sensitivity_summaries
)

multi_sensitivity_episodes = pd.concat(
    sensitivity_episodes,
    ignore_index=True
)

display(multi_sensitivity_summary)

,PanicThreshold,StabilizationThreshold,NumberOfEpisodes,AssetsWithEpisodes,MeanRecoveryDays,MedianRecoveryDays,MinRecoveryDays,MaxRecoveryDays
0,0.4,0.20,131,23,161.152672,102.0,60,1305
1,0.4,0.25,125,23,177.904000,107.0,60,797
2,0.4,0.30,119,23,208.226891,109.0,60,1792
3,0.5,0.20,111,23,145.144144,93.0,60,1305
4,0.5,0.25,108,23,147.555556,93.0,60,614
5,0.5,0.30,103,23,186.165049,107.0,60,1792
6,0.6,0.20,92,23,144.510870,89.5,60,1305
7,0.6,0.25,90,23,139.211111,89.5,60,614
8,0.6,0.30,87,23,176.367816,100.0,60,1308


In [ ]:
# Make percentages easier to read

multi_sensitivity_display = (
    multi_sensitivity_summary.copy()
)

multi_sensitivity_display["PanicThreshold"] = (
    multi_sensitivity_display["PanicThreshold"] * 100
)

multi_sensitivity_display["StabilizationThreshold"] = (
    multi_sensitivity_display["StabilizationThreshold"] * 100
)

multi_sensitivity_display = (
    multi_sensitivity_display
    .sort_values(
        ["PanicThreshold", "StabilizationThreshold"]
    )
    .reset_index(drop=True)
)

display(multi_sensitivity_display.round(2))

,PanicThreshold,StabilizationThreshold,NumberOfEpisodes,AssetsWithEpisodes,MeanRecoveryDays,MedianRecoveryDays,MinRecoveryDays,MaxRecoveryDays
0,40.0,20.0,131,23,161.15,102.0,60,1305
1,40.0,25.0,125,23,177.90,107.0,60,797
2,40.0,30.0,119,23,208.23,109.0,60,1792
3,50.0,20.0,111,23,145.14,93.0,60,1305
4,50.0,25.0,108,23,147.56,93.0,60,614
5,50.0,30.0,103,23,186.17,107.0,60,1792
6,60.0,20.0,92,23,144.51,89.5,60,1305
7,60.0,25.0,90,23,139.21,89.5,60,614
8,60.0,30.0,87,23,176.37,100.0,60,1308


In [ ]:
# Verify that the baseline 50% / 25% specification
# reproduces the finalized episode dataset

baseline_check = multi_sensitivity_summary[
    (multi_sensitivity_summary["PanicThreshold"] == 0.50) &
    (multi_sensitivity_summary["StabilizationThreshold"] == 0.25)
]

display(baseline_check)

baseline_count = int(
    baseline_check["NumberOfEpisodes"].iloc[0]
)

print("Sensitivity baseline episode count:", baseline_count)
print("Saved 03g baseline episode count:", len(baseline_episodes))
print("Expected finalized baseline count:", 108)

print(
    "Counts match:",
    baseline_count == len(baseline_episodes)
)

print(
    "Final baseline count is 108:",
    baseline_count == 108
)

,PanicThreshold,StabilizationThreshold,NumberOfEpisodes,AssetsWithEpisodes,MeanRecoveryDays,MedianRecoveryDays,MinRecoveryDays,MaxRecoveryDays
4,0.5,0.25,108,23,147.555556,93.0,60,614


Sensitivity baseline episode count: 108
Saved 03g baseline episode count: 108
Expected finalized baseline count: 108
Counts match: True
Final baseline count is 108: True


In [ ]:
# Evaluate stability of episode identification across threshold choices
# by comparing panic starts with the finalized 50% / 25% baseline

baseline_episodes["PanicStart"] = pd.to_datetime(
    baseline_episodes["PanicStart"]
)

baseline_keys = set(
    zip(
        baseline_episodes["Asset"],
        baseline_episodes["PanicStart"]
    )
)

overlap_results = []

for panic_threshold, stabilization_threshold in threshold_combinations:

    current = multi_sensitivity_episodes[
        (multi_sensitivity_episodes["PanicThreshold"] == panic_threshold) &
        (
            multi_sensitivity_episodes["StabilizationThreshold"]
            == stabilization_threshold
        )
    ]

    current_keys = set(
        zip(
            current["Asset"],
            current["PanicStart"]
        )
    )

    intersection = len(
        baseline_keys.intersection(current_keys)
    )

    union = len(
        baseline_keys.union(current_keys)
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    baseline_retained = (
        intersection / len(baseline_keys)
        if len(baseline_keys) > 0
        else np.nan
    )

    overlap_results.append({
        "PanicThreshold": panic_threshold,
        "StabilizationThreshold": stabilization_threshold,
        "Episodes": len(current),
        "BaselineEpisodesAlsoIdentified": intersection,
        "BaselineEpisodeRetention": baseline_retained,
        "JaccardSimilarity": jaccard
    })


overlap_df = pd.DataFrame(overlap_results)

overlap_df["PanicThresholdPct"] = (
    overlap_df["PanicThreshold"] * 100
)

overlap_df["StabilizationThresholdPct"] = (
    overlap_df["StabilizationThreshold"] * 100
)

display(
    overlap_df[
        [
            "PanicThresholdPct",
            "StabilizationThresholdPct",
            "Episodes",
            "BaselineEpisodesAlsoIdentified",
            "BaselineEpisodeRetention",
            "JaccardSimilarity"
        ]
    ].round(3)
)

,PanicThresholdPct,StabilizationThresholdPct,Episodes,BaselineEpisodesAlsoIdentified,BaselineEpisodeRetention,JaccardSimilarity
0,40.0,20.0,131,98,0.907,0.695
1,40.0,25.0,125,95,0.880,0.688
2,40.0,30.0,119,90,0.833,0.657
3,50.0,20.0,111,108,1.000,0.973
4,50.0,25.0,108,108,1.000,1.000
5,50.0,30.0,103,103,0.954,0.954
6,60.0,20.0,92,86,0.796,0.754
7,60.0,25.0,90,86,0.796,0.768
8,60.0,30.0,87,84,0.778,0.757


## S&P 500 Threshold Sensitivity

In [ ]:
# Summarize threshold sensitivity for the S&P 500

sp500_sensitivity = []

for panic_threshold, stabilization_threshold in threshold_combinations:

    current = multi_sensitivity_episodes[
        (multi_sensitivity_episodes["Asset"] == "^GSPC") &
        (multi_sensitivity_episodes["PanicThreshold"] == panic_threshold) &
        (
            multi_sensitivity_episodes["StabilizationThreshold"]
            == stabilization_threshold
        )
    ]

    sp500_sensitivity.append({
        "PanicThreshold": panic_threshold,
        "StabilizationThreshold": stabilization_threshold,
        "NumberOfEpisodes": len(current),
        "MeanRecoveryDays": (
            current["RecoveryDays"].mean()
            if not current.empty else np.nan
        ),
        "MedianRecoveryDays": (
            current["RecoveryDays"].median()
            if not current.empty else np.nan
        )
    })


sp500_sensitivity_summary = pd.DataFrame(
    sp500_sensitivity
)

# Percentage columns for presentation
sp500_sensitivity_summary["PanicThresholdPct"] = (
    sp500_sensitivity_summary["PanicThreshold"] * 100
)

sp500_sensitivity_summary["StabilizationThresholdPct"] = (
    sp500_sensitivity_summary["StabilizationThreshold"] * 100
)

display(
    sp500_sensitivity_summary[
        [
            "PanicThresholdPct",
            "StabilizationThresholdPct",
            "NumberOfEpisodes",
            "MeanRecoveryDays",
            "MedianRecoveryDays"
        ]
    ].round(2)
)

,PanicThresholdPct,StabilizationThresholdPct,NumberOfEpisodes,MeanRecoveryDays,MedianRecoveryDays
0,40.0,20.0,7,153.14,91.0
1,40.0,25.0,7,166.00,91.0
2,40.0,30.0,7,183.14,108.0
3,50.0,20.0,5,163.40,108.0
4,50.0,25.0,5,181.40,108.0
5,50.0,30.0,5,205.40,205.0
6,60.0,20.0,4,183.00,159.5
7,60.0,25.0,4,205.50,204.5
8,60.0,30.0,4,205.50,204.5
